In [ ]:
!pip install flash-attn sympy math_verify pylatexenc

In [ ]:
import os

os.environ["VLLM_USE_V1"] = "0"  # use V0 engine of vLLM. in V1 engine model weights cannot be directly access

In [ ]:
from transformers import PreTrainedModel, AutoModelForCausalLM, AutoTokenizer
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from vllm import LLM, SamplingParams
from unittest.mock import patch
import torch
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
from pandas import DataFrame
from tests.adapters import run_get_response_log_probs, run_tokenize_prompt_and_output, run_grpo_microbatch_train_step, run_compute_group_normalized_rewards
from drgrpo_grader import r1_zero_reward_fn
from transformers import PreTrainedTokenizerBase
from torch import Tensor


In [ ]:
def init_vllm(model_id: str, device: str, seed: int, gpu_memory_utilization: float = 0.85):
    """
    Start the inference process, here we use vLLM to hold a model on
    a GPU separate from the policy.
    """
    vllm_set_random_seed(seed)

    # Monkeypatch from TRL:
    # https://github.com/huggingface/trl/blob/22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py
    # Patch vLLM to make sure we can
    # (1) place the vLLM model on the desired device (world_size_patch) and
    # (2) avoid a test that is not designed for our setting (profiling_patch).
    world_size_patch = patch("torch.distributed.get_world_size", return_value=1)
    profiling_patch = patch(
    "vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling",
    return_value=None
    )
    with world_size_patch, profiling_patch:
        return LLM(
            model=model_id,
            device=device,
            dtype=torch.bfloat16,
            enable_prefix_caching=True,
            gpu_memory_utilization=gpu_memory_utilization,
        )

def load_policy_into_vllm_instance(policy: PreTrainedModel, llm: LLM):
    """
    Copied from https://github.com/huggingface/trl/blob/22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py#L670.
    """
    state_dict = policy.state_dict()
    llm_model = llm.llm_engine.model_executor.driver_worker.model_runner.model
    llm_model.load_weights(state_dict.items())

In [ ]:
from typing import Literal

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
model_id = 'Qwen/Qwen2.5-Math-1.5B'
device_0 = 'cuda:0'
device_1 = 'cuda:1'
device_2 = 'cuda:2' # TODO maybe for old policy log probs

n_grpo_steps: int = 200
learning_rate: float = 1e-5
advantage_eps: float = 1e-6
rollout_batch_size: int = 256
group_size: int = 8
sampling_temperature: float = 1.0
sampling_min_tokens: int = 4 # As in Expiter, disallow empty string responses
sampling_max_tokens: int = 1024
epochs_per_rollout_batch: int = 1 # On-policy
train_batch_size: int = 256 # On-policy
gradient_accumulation_steps: int = train_batch_size // group_size # 256 // 8 = 32
gpu_memory_utilization: float = 0.45
loss_type: Literal[
    "no_baseline",
    "reinforce_with_baseline",
    "grpo_clip",
] = "reinforce_with_baseline"
use_std_normalization: bool = True


In [ ]:
assert train_batch_size % gradient_accumulation_steps == 0, (
    "train_batch_size must be divisible by gradient_accumulation_steps"
)
micro_train_batch_size = train_batch_size // gradient_accumulation_steps # micro_train_batch_size = 2
assert rollout_batch_size % group_size == 0, (
    "rollout_batch_size must be divisible by group_size"
)
n_prompts_per_rollout_batch = rollout_batch_size // group_size # n_prompts_per_rollout_batch = 32
assert train_batch_size >= group_size, (
    "train_batch_size must be greater than or equal to group_size"
)
n_microbatches_per_rollout_batch = rollout_batch_size // micro_train_batch_size # n_microbatches_per_rollout_batch = 128

In [ ]:
llm = init_vllm(model_id, device_0, seed, gpu_memory_utilization)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
).to(device_0)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.0,
    betas=(0.9, 0.95),
)

In [ ]:
# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=sampling_temperature, top_p=1.0, min_tokens=sampling_min_tokens, max_tokens=sampling_max_tokens, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

In [ ]:
# Copied from adapters.py, without `labels` and adjust response_mask accordingly
def tokenize_prompt_and_output(
    prompt_strs: list[str],
    output_strs: list[str],
    tokenizer: PreTrainedTokenizerBase,
    device: str
) -> tuple[Tensor, Tensor]:
    input_ids_list = []
    mask_list = []
    max_len = 0

    padding_token_id = tokenizer.encode('<|endoftext|>')[0] # for this model

    # Concatenate first, next find out the max len, then padding accordingly
    for prompt, output in zip(prompt_strs, output_strs):
        prompt_encoded = tokenizer.encode(prompt)
        output_encoded = tokenizer.encode(output)
        full = prompt_encoded + output_encoded
        input_ids_list.append(full)
        mask_list.append([False] * len(prompt_encoded) + [True] * len(output_encoded))
        max_len = max(max_len, len(full))

    for i in range(len(input_ids_list)):
        input = input_ids_list[i]
        mask = mask_list[i]

        input_ids_list[i] = input + [padding_token_id] * (max_len - len(input))
        mask_list[i] = mask + [False] * (max_len - len(mask))

    input_ids = Tensor(input_ids_list).long().to(device)
    response_masks = Tensor(mask_list).to(device)
    return input_ids, response_masks

In [ ]:
def gen_rollout(vllm, df: DataFrame, n:int = n_prompts_per_rollout_batch, size: int = group_size) -> tuple[list[str], list[str], list[str]]:
    sample_set = df.sample(n=n, random_state=seed)
    questions = sample_set['problem'].tolist()
    groud_truth_responses = sample_set['solution'].tolist()
    prompts = [f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""" for question in questions]
    repeated_prompts = [p for p in prompts for _ in range(size)]
    repeated_ground_truths = [r for r in groud_truth_responses for _ in range(size)]
    sampling_params.n = size
    outputs = vllm.generate(prompts, sampling_params)
    rollout_responses = []
    for output in outputs:
        for i in range(size):
            rollout_responses.append(output.outputs[i].text)

    return repeated_prompts, rollout_responses, repeated_ground_truths

In [ ]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  eval_sampling_params.n = 1
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  output_rewards = []

  for index, output in enumerate(outputs):
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    output_rewards.append(rewards)

  return output_rewards

def eval_validation_set(df: DataFrame):
    total_format_reward = 0.
    total_answer_reward = 0.
    total_reward = 0.
    prompts = []
    answers = []

    for index, row in tqdm(enumerate(df.sample(frac=1, random_state=seed).itertuples())):
        question = row.problem
        answer = row.solution
        prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
    User: {question}
    Assistant: <think>"""
        prompts.append(prompt)
        answers.append(answer)

    rewards = evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params)
    for reward in rewards:
        if reward['format_reward'] == 1.0:
            total_format_reward += 1
        if reward['answer_reward'] == 1.0:
            total_answer_reward += 1
        if reward['reward'] == 1.0:
            total_reward += 1

    print(f"Total format reward: {total_format_reward}")
    print(f"Total answer reward: {total_answer_reward}")
    print(f"Total reward: {total_reward}")

In [ ]:
# Copied from adapters.py but with partial reward on format

def compute_group_normalized_rewards(
    reward_fn: Callable,
    rollout_responses: list[str],
    repeated_ground_truths: list[str],
    group_size: int,
    advantage_eps: float,
    normalize_by_std: bool,
) -> tuple[torch.Tensor, torch.Tensor, dict[str, float]]:
    # Compute raw rewards
    raw_rewards = [1.0 if reward_fn(response, truth)['reward'] == 1.0 else (0.2 if reward_fn(response, truth)['format_reward'] == 1.0 else 0.0)
                   for response, truth in zip(rollout_responses, repeated_ground_truths)]

    raw_rewards = torch.tensor(raw_rewards, dtype=torch.float32)

    # Reshape into groups for easier processing
    rewards_grouped = raw_rewards.view(-1, group_size)  # Shape: (n_groups, group_size)

    # Compute group means and center each group
    group_means = rewards_grouped.mean(dim=1, keepdim=True)  # Shape: (n_groups, 1)
    normalized_rewards = rewards_grouped - group_means

    # Normalize by standard deviation if requested
    if normalize_by_std:
        group_stds = torch.std(rewards_grouped, dim=1, keepdim=True)  # Sample std
        normalized_rewards = normalized_rewards / (group_stds + advantage_eps)

    # Flatten back to original shape
    normalized_rewards = normalized_rewards.flatten()

    # Create metadata
    metadata = {
        'mean_raw_reward': float(raw_rewards.mean()),
        'std_raw_reward': float(raw_rewards.std()),
        'mean_normalized_reward': float(normalized_rewards.mean()),
        'std_normalized_reward': float(normalized_rewards.std()),
    }

    return normalized_rewards, raw_rewards, metadata

In [ ]:
df_test = pd.read_parquet("math_12k_test.parquet")
df_train = pd.read_parquet("math_12k_train.parquet")

global_it = 0
for step in range(n_grpo_steps):
    model.train()
    repeated_prompts, rollout_responses, repeated_ground_truths = gen_rollout(llm, df_train, n_prompts_per_rollout_batch, group_size)

    # training
    for idx in range(0, len(repeated_prompts), group_size):
        mini_batch_prompts = repeated_prompts[idx : idx + group_size]
        mini_batch_rollout_responses = rollout_responses[idx : idx + group_size]
        mini_batch_ground_truths = repeated_ground_truths[idx : idx + group_size]

        normalized_rewards, raw_rewards, _ = compute_group_normalized_rewards(r1_zero_reward_fn, mini_batch_rollout_responses, mini_batch_ground_truths, group_size, advantage_eps, use_std_normalization)
        normalized_rewards = normalized_rewards.unsqueeze(-1).to(device_0)
        
        input_ids, response_masks = tokenize_prompt_and_output(mini_batch_prompts, mini_batch_rollout_responses, tokenizer, device_0)
        policy_log_probs = run_get_response_log_probs(model, input_ids, input_ids, False)['log_probs'] # Reuse the function, ignore `labels` part, should be fine

        # print(f"{policy_log_probs=}, {response_masks=}, {normalized_rewards=}")
        
        loss, _ = run_grpo_microbatch_train_step(policy_log_probs=policy_log_probs, response_mask=response_masks, gradient_accumulation_steps=gradient_accumulation_steps, loss_type=loss_type, advantages=normalized_rewards)

        print({
            "loss": loss.item(),
            "raw_rewards": raw_rewards.sum().item(),
            "mask_nonzero": int(response_masks.sum().item()),
            "adv_mean": float(normalized_rewards.mean().item()),
            "adv_std": float(normalized_rewards.std().item()),
            "logp_mean": float(policy_log_probs.mean().item()),
        })
        
        global_it += 1
        if (global_it % gradient_accumulation_steps) == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

    load_policy_into_vllm_instance(model, llm)

    # validation
    if (step + 1) % 10 == 0:
        print(f"Step {step:06d}:")
        # reload model into vllm and eval
        eval_validation_set(df_test)
